# A. Purpose of experiment

Manual **Run All** xLSTM depth study replacing the scientific role of the legacy BiGRU depth study. Only the MagNet representation-pretraining stage is executable now. The test split is never accessed. The downstream confirmation remains blocked until real fine-tuning data are available.

**Saved-output notice:** Existing outputs in this file predate the v1.1 migration and are retained only as historical execution evidence. Run All replaces them with current v1.1 validation output.


In [5]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
RUN_DEPTHS = [2, 4, 6, 8, 10]
FORCE_RETRAIN = False
AUTO_RESUME = True
print('PASS — manual controls loaded')
print(f'RUN_DEPTHS={RUN_DEPTHS}')
print(f'FORCE_RETRAIN={FORCE_RETRAIN} (applies only to depths explicitly listed in RUN_DEPTHS)')
print(f'AUTO_RESUME={AUTO_RESUME}')
output_root = PROJECT_ROOT / 'experiments/xlstm_depth_v1'
has_results = output_root.exists() and any(output_root.rglob('training_summary.json'))
print(f"DEPTH EXPERIMENT STATUS: {'RECOVERY/COMPLETED ARTIFACTS FOUND' if has_results else 'NEW RUN'}")


PASS — manual controls loaded
RUN_DEPTHS=[2, 4, 6, 8, 10]
FORCE_RETRAIN=False (applies only to depths explicitly listed in RUN_DEPTHS)
AUTO_RESUME=True
DEPTH EXPERIMENT STATUS: NEW RUN


# B. Environment verification


In [6]:
import sys
import os
import numpy as np
import torch
REQUIRED_INTERPRETER = os.environ.get('MEPI_REQUIRED_INTERPRETER')
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'UNAVAILABLE'
print(f'sys.executable: {sys.executable}')
print(f'Python version: {sys.version.replace(chr(10), chr(32))}')
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA build: {torch.version.cuda}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU name: {gpu_name}')
print(f'NumPy version: {np.__version__}')
if REQUIRED_INTERPRETER and str(Path(sys.executable).resolve()) != str(Path(REQUIRED_INTERPRETER).resolve()):
    print('FAIL — wrong interpreter')
    raise RuntimeError('Select Python (trans-core), restart the kernel, then Run All')
if not torch.cuda.is_available():
    print('FAIL — CUDA is required for the scientific run')
    raise RuntimeError('CUDA is unavailable in Python (trans-core); stop before training')
print('PASS — required Python (trans-core) interpreter and CUDA are active')


sys.executable: <USER_HOME>/miniconda3/envs/trans-core/bin/python
Python version: 3.11.15 | packaged by conda-forge | (main, Aug 11 2026, 10:26:29) [GCC 14.4.0]
PyTorch version: 2.5.1
CUDA build: 12.1
CUDA available: True
GPU name: NVIDIA GeForce RTX 3050
NumPy version: 2.4.6
PASS — required Python (trans-core) interpreter and CUDA are active


In [7]:
from pathlib import Path
import sys

# Locate MEPI project root robustly whether the notebook is launched
# from the repository root or from notebooks/.
cwd = Path.cwd().resolve()

candidates = [cwd, *cwd.parents]
PROJECT_ROOT = next(
    (p for p in candidates if (p / "src" / "mepi_v1").is_dir()),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError(
        f"Could not locate MEPI project root from working directory: {cwd}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Working directory :", cwd)
print("PROJECT_ROOT      :", PROJECT_ROOT)
print("Python path entry :", sys.path[0])

assert (PROJECT_ROOT / "src" / "mepi_v1").is_dir()
print("[PASS] MEPI project source path configured")

Working directory : <REPOSITORY_ROOT>/notebooks
PROJECT_ROOT      : <REPOSITORY_ROOT>
Python path entry : <REPOSITORY_ROOT>
[PASS] MEPI project source path configured


# C. Frozen-protocol verification


In [8]:
from src.mepi_v1.reproducibility import sha256_file
EXPECTED_PROTOCOL_SHA256 = 'a515052cd2f8cf2970b731cfb48dee055c316a5d4acc0f0ca90313436c344c0f'
protocol_paths = [PROJECT_ROOT / 'MEPI-FROZEN-PROTOCOL v1.1.md', PROJECT_ROOT / 'docs/MEPI-FROZEN-PROTOCOL v1.1.md']
protocol_hashes = [sha256_file(path) for path in protocol_paths]
passed = protocol_hashes == [EXPECTED_PROTOCOL_SHA256, EXPECTED_PROTOCOL_SHA256]
print(f"{'PASS' if passed else 'FAIL'} — frozen protocol hashes: {protocol_hashes}")
if not passed:
    raise RuntimeError('Frozen protocol is missing or changed; training is blocked')


PASS — frozen protocol hashes: ['ef8f1f85c94ddc18645d25f4537a3eed45e8f46c6d900f419ff3c8f36be99cef', 'ef8f1f85c94ddc18645d25f4537a3eed45e8f46c6d900f419ff3c8f36be99cef']


# D. Dataset and split verification


In [9]:
from src.mepi_v1.xlstm_depth import XLSTMDepthRun
probe = XLSTMDepthRun(PROJECT_ROOT, 2)
dataset_check = probe.validate_dataset()
assert dataset_check['waveform_shape'][1] == 1024
assert dataset_check['test_split_accessed'] is False
print('PASS — B(t) has 1024 points; only train and validation manifests were inspected')


[PASS] Depth-study dataset: <REPOSITORY_ROOT>/data/raw/source_datasets/Dataset/magnet_pretrain_186k.h5
[PASS] Train/validation counts: {'train': 149405, 'validation': 18676}
[PASS] Test manifest and test dataset not opened
PASS — B(t) has 1024 points; only train and validation manifests were inspected


# E. Dataset and split fingerprints


In [10]:
EXPECTED_DATASET_SHA256 = '0aca43184e9bd6cc90eec736a89fe1ea32f45b126527cb97918cd92684ac2c19'
EXPECTED_TRAIN_SHA256 = '2e06050bbe1cff8b6f4f7ae3a99788d6914e214353a5e6a2f2ece7463b6377b0'
EXPECTED_VALIDATION_SHA256 = 'aa7d1ee30ca3f3967eadee81d2a31e0a9bcb7e0bc5421f36cb849d6da1dc9749'
prepared = probe._prepare()
actual = {
    'dataset': prepared['compatibility']['dataset_fingerprint'],
    'train': sha256_file(prepared['manifests']['train']),
    'validation': sha256_file(prepared['manifests']['validation']),
}
expected = {'dataset': EXPECTED_DATASET_SHA256, 'train': EXPECTED_TRAIN_SHA256, 'validation': EXPECTED_VALIDATION_SHA256}
print(f"{'PASS' if actual == expected else 'FAIL'} — fingerprints: {actual}")
if actual != expected:
    raise RuntimeError('Frozen dataset or train/validation manifests changed')
print('PASS — MagNet test manifest was not read or fingerprinted')


PASS — fingerprints: {'dataset': '0aca43184e9bd6cc90eec736a89fe1ea32f45b126527cb97918cd92684ac2c19', 'train': '2e06050bbe1cff8b6f4f7ae3a99788d6914e214353a5e6a2f2ece7463b6377b0', 'validation': 'aa7d1ee30ca3f3967eadee81d2a31e0a9bcb7e0bc5421f36cb849d6da1dc9749'}
PASS — MagNet test manifest was not read or fingerprinted


# F. xLSTM baseline architecture


In [11]:
from src.mepi_v1.backbones import XLSTMBackbone
from src.mepi_v1.models import PretrainingModel
baseline = PretrainingModel('xLSTM', material_count=10, latent_dim=256, backbone_layers=2)
assert isinstance(baseline.backbone, XLSTMBackbone)
print('PASS — waveform encoder -> operating encoder -> fusion -> stacked xLSTM blocks -> temporary material-routed heads')
print('PASS — material identity is not a predictive input; PIRL and MTPH are absent')


PASS — waveform encoder -> operating encoder -> fusion -> stacked xLSTM blocks -> temporary material-routed heads
PASS — material identity is not a predictive input; PIRL and MTPH are absent


# G. Controlled variables


In [12]:
fixed = {
 'waveform_length': 1024, 'predictive_tabular': ['frequency', 'temperature'],
 'latent_dim': 256, 'epochs': 10, 'batch_size': probe.config['training']['batch_size'],
 'optimizer': 'AdamW', 'learning_rate': probe.config['training']['learning_rate'],
 'scheduler': 'constant LambdaLR', 'dataloader': {k: probe.config['training'].get(k) for k in ['num_workers','pin_memory','persistent_workers','prefetch_factor','cache_mode']},
 'amp': probe.config['training']['amp'], 'tf32': probe.config['runtime']['allow_tf32'],
 'seed': probe.seed, 'target_transform': prepared['preprocessing']['target'],
}
print('PASS — depth is the only varied model/configuration value')
fixed


PASS — depth is the only varied model/configuration value


{'waveform_length': 1024,
 'predictive_tabular': ['frequency', 'temperature'],
 'latent_dim': 256,
 'epochs': 10,
 'batch_size': 256,
 'optimizer': 'AdamW',
 'learning_rate': 3e-05,
 'scheduler': 'constant LambdaLR',
 'dataloader': {'num_workers': 4,
  'pin_memory': True,
  'persistent_workers': True,
  'prefetch_factor': 2,
  'cache_mode': 'hdf5'},
 'amp': True,
 'tf32': True,
 'seed': 42,
 'target_transform': {'transform': 'log1p_standardize',
  'scaler': 'population mean and standard deviation',
  'inverse': 'expm1(z * scale + mean)',
  'state': {'mean': [11.073220047200586],
   'scale': [1.849061097213353],
   'fitted_split': 'train',
   'sample_count': 149405}}}

# H. Depth candidates [2, 4, 6, 8, 10]


In [13]:
from src.mepi_v1.xlstm_depth import DEPTHS
invalid = sorted(set(RUN_DEPTHS) - set(DEPTHS))
print(f"{'PASS' if not invalid else 'FAIL'} — canonical DEPTHS={list(DEPTHS)}; requested RUN_DEPTHS={RUN_DEPTHS}")
if invalid:
    raise ValueError(f'Unsupported depth values: {invalid}')


PASS — canonical DEPTHS=[2, 4, 6, 8, 10]; requested RUN_DEPTHS=[2, 4, 6, 8, 10]


# I. Parameter-count preview


In [14]:
import pandas as pd
preview = []
for depth in DEPTHS:
    model = PretrainingModel('xLSTM', material_count=10, latent_dim=256, backbone_layers=depth)
    preview.append({'depth': depth, 'parameter_count': sum(p.numel() for p in model.parameters()), 'block_count': len(model.backbone.blocks)})
preview_df = pd.DataFrame(preview)
assert preview_df['parameter_count'].is_monotonic_increasing
print('PASS — block count equals depth and parameter count increases with depth')
preview_df


PASS — block count equals depth and parameter count increases with depth


,depth,parameter_count,block_count
0,2,2019338,2
1,4,3336202,4
2,6,4653066,6
3,8,5969930,8
4,10,7286794,10


# J. Training setup

Each candidate has its own visible cell below. Compatible completed candidates are skipped; interrupted candidates resume automatically from `last_checkpoint.pt`.


In [15]:
import torch
from src.mepi_v1.xlstm_depth import official_pretraining_snapshot
official_before = official_pretraining_snapshot(PROJECT_ROOT)
depth_results = {}
def run_manual_depth(depth):
    if depth not in RUN_DEPTHS:
        print(f'Depth {depth}/10: SKIPPED BY RUN_DEPTHS')
        return None
    run = XLSTMDepthRun(PROJECT_ROOT, depth, force_retrain=FORCE_RETRAIN, auto_resume=AUTO_RESUME)
    status = run.inspect_status()
    if status['status'] == 'COMPLETE':
        summary = run.validate_completed_compatibility()
        print(f'Depth {depth}/10: COMPLETE — skipped safely')
        depth_results[depth] = summary
        return summary
    print(f"Depth {depth}/10: {'RESUME' if status['status'] == 'INCOMPLETE' else 'FROM SCRATCH'}")
    run.check_environment(require_cuda=True)
    run.validate_source_protocol()
    run.validate_dataset()
    run.construct_model_smoke()
    run.inspect_checkpoint()
    torch.cuda.reset_peak_memory_stats()
    run.train_or_resume()
    summary = run.finalize_validation_only(peak_gpu_memory_bytes=int(torch.cuda.max_memory_allocated()))
    run.write_depth_evidence(summary, official_before)
    if official_pretraining_snapshot(PROJECT_ROOT) != official_before:
        raise RuntimeError('Official pretraining evidence changed during the depth run')
    print(f"PASS — Depth {depth} complete; validation MAE_norm={summary['validation_metrics']['normalized_mae']}")
    print(f"Checkpoint saved: {summary['best_checkpoint']}")
    depth_results[depth] = summary
    return summary
print('PASS — manual training helper ready; no candidate has been launched by this setup cell')


PASS — manual training helper ready; no candidate has been launched by this setup cell


# K. Depth-2 training


In [16]:
depth_2_summary = run_manual_depth(2)


STATUS: NEW RUN
Depth 2/10: FROM SCRATCH
[PASS] Project root: <REPOSITORY_ROOT>
[PASS] Python executable: <USER_HOME>/miniconda3/envs/trans-core/bin/python
[PASS] Python version: 3.11.15 | packaged by conda-forge | (main, Aug 11 2026, 10:26:29) [GCC 14.4.0]
[PASS] PyTorch import: 2.5.1
[PASS] PyTorch CUDA version: 12.1
[PASS] CUDA available: True
[PASS] GPU model: NVIDIA GeForce RTX 3050
[PASS] Python (trans-core) kernel interpreter: display='Python (trans-core)'; kernel=<USER_HOME>/miniconda3/envs/trans-core/bin/python3.11; active=<USER_HOME>/miniconda3/envs/trans-core/bin/python3.11
[PASS] Frozen protocol found: <REPOSITORY_ROOT>/docs/MEPI_FROZEN_PROTOCOL_v1.0.md
[PASS] src.mepi_v1 imports: source fingerprint 14bc064a53f22849c3723254a57d6604a3b14339bf987c22d2821cffeefd3816
[PASS] All eight candidate configs share one non-architecture setup: 05e4f2864bbbcaf68de732438a869f75bab0273e646166a669686f3694ab67a1
Git SHA: UNAVAILABLE (source-tree fingerprint retained)
[PASS] Depth-study datas

# L. Depth-4 training


In [17]:
depth_4_summary = run_manual_depth(4)


STATUS: NEW RUN
Depth 4/10: FROM SCRATCH
[PASS] Project root: <REPOSITORY_ROOT>
[PASS] Python executable: <USER_HOME>/miniconda3/envs/trans-core/bin/python
[PASS] Python version: 3.11.15 | packaged by conda-forge | (main, Aug 11 2026, 10:26:29) [GCC 14.4.0]
[PASS] PyTorch import: 2.5.1
[PASS] PyTorch CUDA version: 12.1
[PASS] CUDA available: True
[PASS] GPU model: NVIDIA GeForce RTX 3050
[PASS] Python (trans-core) kernel interpreter: display='Python (trans-core)'; kernel=<USER_HOME>/miniconda3/envs/trans-core/bin/python3.11; active=<USER_HOME>/miniconda3/envs/trans-core/bin/python3.11
[PASS] Frozen protocol found: <REPOSITORY_ROOT>/docs/MEPI_FROZEN_PROTOCOL_v1.0.md
[PASS] src.mepi_v1 imports: source fingerprint 14bc064a53f22849c3723254a57d6604a3b14339bf987c22d2821cffeefd3816
[PASS] All eight candidate configs share one non-architecture setup: 05e4f2864bbbcaf68de732438a869f75bab0273e646166a669686f3694ab67a1
Git SHA: UNAVAILABLE (source-tree fingerprint retained)
[PASS] Depth-study datas

# M. Depth-6 training


In [18]:
depth_6_summary = run_manual_depth(6)


STATUS: NEW RUN
Depth 6/10: FROM SCRATCH
[PASS] Project root: <REPOSITORY_ROOT>
[PASS] Python executable: <USER_HOME>/miniconda3/envs/trans-core/bin/python
[PASS] Python version: 3.11.15 | packaged by conda-forge | (main, Aug 11 2026, 10:26:29) [GCC 14.4.0]
[PASS] PyTorch import: 2.5.1
[PASS] PyTorch CUDA version: 12.1
[PASS] CUDA available: True
[PASS] GPU model: NVIDIA GeForce RTX 3050
[PASS] Python (trans-core) kernel interpreter: display='Python (trans-core)'; kernel=<USER_HOME>/miniconda3/envs/trans-core/bin/python3.11; active=<USER_HOME>/miniconda3/envs/trans-core/bin/python3.11
[PASS] Frozen protocol found: <REPOSITORY_ROOT>/docs/MEPI_FROZEN_PROTOCOL_v1.0.md
[PASS] src.mepi_v1 imports: source fingerprint 14bc064a53f22849c3723254a57d6604a3b14339bf987c22d2821cffeefd3816
[PASS] All eight candidate configs share one non-architecture setup: 05e4f2864bbbcaf68de732438a869f75bab0273e646166a669686f3694ab67a1
Git SHA: UNAVAILABLE (source-tree fingerprint retained)
[PASS] Depth-study datas

# N. Depth-8 training


In [19]:
depth_8_summary = run_manual_depth(8)


STATUS: NEW RUN
Depth 8/10: FROM SCRATCH
[PASS] Project root: <REPOSITORY_ROOT>
[PASS] Python executable: <USER_HOME>/miniconda3/envs/trans-core/bin/python
[PASS] Python version: 3.11.15 | packaged by conda-forge | (main, Aug 11 2026, 10:26:29) [GCC 14.4.0]
[PASS] PyTorch import: 2.5.1
[PASS] PyTorch CUDA version: 12.1
[PASS] CUDA available: True
[PASS] GPU model: NVIDIA GeForce RTX 3050
[PASS] Python (trans-core) kernel interpreter: display='Python (trans-core)'; kernel=<USER_HOME>/miniconda3/envs/trans-core/bin/python3.11; active=<USER_HOME>/miniconda3/envs/trans-core/bin/python3.11
[PASS] Frozen protocol found: <REPOSITORY_ROOT>/docs/MEPI_FROZEN_PROTOCOL_v1.0.md
[PASS] src.mepi_v1 imports: source fingerprint 14bc064a53f22849c3723254a57d6604a3b14339bf987c22d2821cffeefd3816
[PASS] All eight candidate configs share one non-architecture setup: 05e4f2864bbbcaf68de732438a869f75bab0273e646166a669686f3694ab67a1
Git SHA: UNAVAILABLE (source-tree fingerprint retained)
[PASS] Depth-study datas

# O. Depth-10 training


In [20]:
depth_10_summary = run_manual_depth(10)


STATUS: NEW RUN
Depth 10/10: FROM SCRATCH
[PASS] Project root: <REPOSITORY_ROOT>
[PASS] Python executable: <USER_HOME>/miniconda3/envs/trans-core/bin/python
[PASS] Python version: 3.11.15 | packaged by conda-forge | (main, Aug 11 2026, 10:26:29) [GCC 14.4.0]
[PASS] PyTorch import: 2.5.1
[PASS] PyTorch CUDA version: 12.1
[PASS] CUDA available: True
[PASS] GPU model: NVIDIA GeForce RTX 3050
[PASS] Python (trans-core) kernel interpreter: display='Python (trans-core)'; kernel=<USER_HOME>/miniconda3/envs/trans-core/bin/python3.11; active=<USER_HOME>/miniconda3/envs/trans-core/bin/python3.11
[PASS] Frozen protocol found: <REPOSITORY_ROOT>/docs/MEPI_FROZEN_PROTOCOL_v1.0.md
[PASS] src.mepi_v1 imports: source fingerprint 14bc064a53f22849c3723254a57d6604a3b14339bf987c22d2821cffeefd3816
[PASS] All eight candidate configs share one non-architecture setup: 05e4f2864bbbcaf68de732438a869f75bab0273e646166a669686f3694ab67a1
Git SHA: UNAVAILABLE (source-tree fingerprint retained)
[PASS] Depth-study data

# P. Validation comparison


In [21]:
from src.mepi_v1.xlstm_depth import summarize_depth_experiment
comparison = summarize_depth_experiment(PROJECT_ROOT)
comparison_df = pd.DataFrame(comparison).rename(columns={'training_runtime_seconds': 'runtime', 'best_checkpoint_sha256': 'checkpoint_sha256'})
display_columns = ['depth','parameter_count','best_epoch','validation_mae_norm','validation_rmse','validation_r2','runtime','peak_gpu_memory_gib','checkpoint_sha256','training_horizon_boundary_reached']
completed_df = comparison_df[comparison_df['status'] == 'COMPLETE'][display_columns].sort_values('validation_mae_norm')
print('PASS — comparison uses validation metrics only and is sorted by validation MAE_norm')
completed_df


PASS — comparison uses validation metrics only and is sorted by validation MAE_norm


,depth,parameter_count,best_epoch,validation_mae_norm,validation_rmse,validation_r2,runtime,peak_gpu_memory_gib,checkpoint_sha256,training_horizon_boundary_reached
0,8,5969930,10,0.020495,0.027949,0.999233,2381.536420,4.153300,fdbeb87932dd6e20c45104f2ff074e267b414587a5ce9e...,True
1,10,7286794,10,0.024326,0.032505,0.998962,2993.800992,4.985282,3f78fba42614bf06c0cd8b36982b183cbd581710fa6034...,True
2,4,3336202,9,0.026314,0.037244,0.998638,1389.462312,2.485726,0570fc6a0feca68fa7bfb67edd1082b97f77411bb12ed8...,False
3,6,4653066,10,0.028504,0.037054,0.998652,1869.379593,3.319738,1b960ee46dc6cf676bfe70894c10b07bb04ab2d3e40f83...,True
4,2,2019338,9,0.034605,0.049907,0.997554,978.738730,1.654724,3b7795b2f9f09b1c0a6d1eab1603fe30892f62c8742d03...,False


# Q. Selected depth


In [22]:
if len(completed_df) == len(DEPTHS):
    selected_depth = int(completed_df.iloc[0]['depth'])
    print(f'SELECTED xLSTM DEPTH = {selected_depth}')
    print('SELECTION CRITERION =\nminimum validation MAE_norm')
    print('TEST DATA ACCESSED =\nNO')
else:
    selected_depth = None
    print(f'PENDING — {len(completed_df)}/{len(DEPTHS)} candidates are complete; no depth selected')


SELECTED xLSTM DEPTH = 8
SELECTION CRITERION =
minimum validation MAE_norm
TEST DATA ACCESSED =
NO


# R. Reproducibility and evidence summary


In [23]:
print('PASS — reproducibility/evidence summary')
print(f'Official pretraining tree SHA-256: {official_before["tree_sha256"]}')
print(f'Dataset SHA-256: {EXPECTED_DATASET_SHA256}')
print(f'Train manifest SHA-256: {EXPECTED_TRAIN_SHA256}')
print(f'Validation manifest SHA-256: {EXPECTED_VALIDATION_SHA256}')
print('Test manifest/dataset access: NO')
print('A best_epoch equal to 10 is reported as TRAINING_HORIZON_BOUNDARY_REACHED=True; no run is extended.')
completed_df


PASS — reproducibility/evidence summary
Official pretraining tree SHA-256: c88efbb07c7d18042e71cc87b3b63c5d0c02d6d7600c34368b35755ab3129265
Dataset SHA-256: 0aca43184e9bd6cc90eec736a89fe1ea32f45b126527cb97918cd92684ac2c19
Train manifest SHA-256: 2e06050bbe1cff8b6f4f7ae3a99788d6914e214353a5e6a2f2ece7463b6377b0
Validation manifest SHA-256: aa7d1ee30ca3f3967eadee81d2a31e0a9bcb7e0bc5421f36cb849d6da1dc9749
Test manifest/dataset access: NO
A best_epoch equal to 10 is reported as TRAINING_HORIZON_BOUNDARY_REACHED=True; no run is extended.


,depth,parameter_count,best_epoch,validation_mae_norm,validation_rmse,validation_r2,runtime,peak_gpu_memory_gib,checkpoint_sha256,training_horizon_boundary_reached
0,8,5969930,10,0.020495,0.027949,0.999233,2381.536420,4.153300,fdbeb87932dd6e20c45104f2ff074e267b414587a5ce9e...,True
1,10,7286794,10,0.024326,0.032505,0.998962,2993.800992,4.985282,3f78fba42614bf06c0cd8b36982b183cbd581710fa6034...,True
2,4,3336202,9,0.026314,0.037244,0.998638,1389.462312,2.485726,0570fc6a0feca68fa7bfb67edd1082b97f77411bb12ed8...,False
3,6,4653066,10,0.028504,0.037054,0.998652,1869.379593,3.319738,1b960ee46dc6cf676bfe70894c10b07bb04ab2d3e40f83...,True
4,2,2019338,9,0.034605,0.049907,0.997554,978.738730,1.654724,3b7795b2f9f09b1c0a6d1eab1603fe30892f62c8742d03...,False


# S. Downstream status


In [24]:
if selected_depth is not None:
    print('MAGNET DEPTH STUDY: COMPLETE')
else:
    print('MAGNET DEPTH STUDY: INCOMPLETE')
print('DOWNSTREAM DEPTH CONFIRMATION:')
print('WAITING FOR REAL FINETUNE DATA')
print('No downstream fine-tuning, loss-weight, final-test, or frequency-screening result was generated here.')


MAGNET DEPTH STUDY: COMPLETE
DOWNSTREAM DEPTH CONFIRMATION:
WAITING FOR REAL FINETUNE DATA
No downstream fine-tuning, loss-weight, final-test, or frequency-screening result was generated here.
